<a href="https://colab.research.google.com/github/Roland-Education/eng-ai-agents/blob/main/assignment3_v1_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

 Assignment 3 — UAV Drone Detection and Tracking


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/assignment3_drone'
print('Project directory:', PROJECT_DIR)


## 2. Install dependencies

In [ ]:
!pip install -q ultralytics filterpy opencv-python-headless yt-dlp huggingface_hub datasets
!apt-get install -q ffmpeg


## 3. Verify GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


## 4. Create project directories

In [ ]:
import os

dirs = [
    PROJECT_DIR,
    PROJECT_DIR + '/videos',
    PROJECT_DIR + '/frames',
    PROJECT_DIR + '/detections',
    PROJECT_DIR + '/output_videos',
    PROJECT_DIR + '/models',
]

for d in dirs:
    os.makedirs(d, exist_ok=True)
    print('OK', d)


## 5. Download test videos

In [ ]:
VIDEOS = {
    'drone_video_1.mp4': 'https://www.youtube.com/watch?v=DhmZ6W1UAv4',
    'drone_video_2.mp4': 'https://www.youtube.com/watch?v=YrydHPwRelI',
}

for filename, url in VIDEOS.items():
    out_path = PROJECT_DIR + '/videos/' + filename
    if os.path.exists(out_path):
        print('Already downloaded, skipping:', filename)
        continue
    print('Downloading', filename)
    os.system(
        'yt-dlp -f "bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]"'
        ' -o "' + out_path + '" "' + url + '"'
    )

print('Done.')


## 6. Extract frames

In [ ]:
import glob

FPS = 5  # adjust if drone moves fast

for video_path in glob.glob(PROJECT_DIR + '/videos/*.mp4'):
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    frame_dir = PROJECT_DIR + '/frames/' + video_name
    os.makedirs(frame_dir, exist_ok=True)

    existing = glob.glob(frame_dir + '/*.jpg')
    if existing:
        print('Already extracted:', video_name, '(' + str(len(existing)) + ' frames)')
        continue

    print('Extracting frames from', video_name)
    os.system(
        'ffmpeg -i "' + video_path + '" -vf "fps=' + str(FPS) + '" "'
        + frame_dir + '/frame_%04d.jpg" -hide_banner -loglevel error'
    )
    count = len(glob.glob(frame_dir + '/*.jpg'))
    print(' ->', count, 'frames extracted')

print('Setup complete. Ready to start Task 1.')


## 7. Task 1 — Drone Detection with YOLOv8

Pretrained YOLOv8n is used as the baseline. Fine-tuned on a
drone-specific dataset from https://universe.roboflow.com/search?q=drone+detection for better accuracy.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
os.system('cp yolov8n.pt "' + PROJECT_DIR + '/models/yolov8n.pt"')
print('Model ready.')


In [ ]:
import shutil
import cv2

# After fine-tuning on a drone dataset, update this set to your actual class name.
# With base COCO weights, 'airplane' is the closest available proxy.
DRONE_CLASS_NAMES = {'drone', 'uav', 'airplane'}
CONF_THRESHOLD = 0.3

detection_log = {}  # { video_name: { frame_path: [box_dicts] } }

for video_path in glob.glob(PROJECT_DIR + '/videos/*.mp4'):
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    frame_dir = PROJECT_DIR + '/frames/' + video_name
    det_dir = PROJECT_DIR + '/detections/' + video_name
    os.makedirs(det_dir, exist_ok=True)

    frame_paths = sorted(glob.glob(frame_dir + '/*.jpg'))
    print('\nRunning detection on', video_name, '-', len(frame_paths), 'frames')

    detection_log[video_name] = {}
    detected_count = 0

    for frame_path in frame_paths:
        results = model(frame_path, conf=CONF_THRESHOLD, verbose=False)
        boxes = []
        for result in results:
            for box in result.boxes:
                class_name = model.names[int(box.cls)].lower()
                if any(d in class_name for d in DRONE_CLASS_NAMES):
                    boxes.append({
                        'xyxy': box.xyxy[0].tolist(),
                        'conf': float(box.conf),
                        'class': class_name,
                    })
        if boxes:
            detection_log[video_name][frame_path] = boxes
            shutil.copy(frame_path, det_dir + '/' + os.path.basename(frame_path))
            detected_count += 1

    print(' ->', detected_count, '/', len(frame_paths), 'frames with detections')

print('\nDetection complete.')


## 8. Task 2 — Kalman Filter Tracking


In [ ]:
import numpy as np
from filterpy.kalman import KalmanFilter


def make_kalman_filter():
    # State: [cx, cy, vx, vy]  Measurement: [cx, cy]
    kf = KalmanFilter(dim_x=4, dim_z=2)
    kf.F = np.array([[1,0,1,0],[0,1,0,1],[0,0,1,0],[0,0,0,1]], dtype=float)
    kf.H = np.array([[1,0,0,0],[0,1,0,0]], dtype=float)
    kf.R = np.eye(2) * 10.0    # measurement noise
    kf.Q = np.eye(4) * 0.1     # process noise
    kf.P = np.eye(4) * 500.0   # initial covariance
    return kf


def box_center(xyxy):
    x1, y1, x2, y2 = xyxy
    return [(x1+x2)/2, (y1+y2)/2]


def run_tracker(video_name, frame_paths, detection_log, max_miss=5):
    kf = None
    trajectory = []
    miss_count = 0
    initialized = False
    det_map = detection_log.get(video_name, {})

    for frame_path in frame_paths:
        boxes = det_map.get(frame_path, [])
        best_box = boxes[0] if boxes else None

        if not initialized:
            if best_box:
                cx, cy = box_center(best_box['xyxy'])
                kf = make_kalman_filter()
                kf.x = np.array([[cx],[cy],[0.],[0.]])
                initialized = True
                miss_count = 0
                trajectory.append((frame_path, best_box, [cx, cy]))
            continue

        kf.predict()

        if best_box:
            cx, cy = box_center(best_box['xyxy'])
            kf.update(np.array([[cx],[cy]]))
            miss_count = 0
        else:
            miss_count += 1
            if miss_count > max_miss:
                initialized = False
                kf = None
                continue

        trajectory.append((frame_path, best_box, [float(kf.x[0]), float(kf.x[1])]))

    return trajectory


trajectories = {}
for video_path in glob.glob(PROJECT_DIR + '/videos/*.mp4'):
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    frame_paths = sorted(glob.glob(PROJECT_DIR + '/frames/' + video_name + '/*.jpg'))
    traj = run_tracker(video_name, frame_paths, detection_log)
    trajectories[video_name] = traj
    print(video_name + ':', len(traj), 'tracked frames')

print('Tracking complete.')


## 9. Produce output videos

In [ ]:
def draw_frame(frame, box, centers_so_far):
    if box:
        x1, y1, x2, y2 = [int(v) for v in box['xyxy']]
        cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
        label = box['class'] + ' ' + str(round(box['conf'], 2))
        cv2.putText(frame, label, (x1, y1-8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
    pts = [(int(c[0]), int(c[1])) for c in centers_so_far]
    for i in range(1, len(pts)):
        cv2.line(frame, pts[i-1], pts[i], (0,0,255), 2)
    if pts:
        cv2.circle(frame, pts[-1], 5, (0,0,255), -1)
    return frame


OUTPUT_FPS = 10

for video_name, traj in trajectories.items():
    if not traj:
        print('No tracked frames for', video_name, '- skipping.')
        continue

    tmp_dir = '/tmp/' + video_name + '_out'
    os.makedirs(tmp_dir, exist_ok=True)
    centers_so_far = []

    for i, (frame_path, box, center) in enumerate(traj):
        centers_so_far.append(center)
        frame = cv2.imread(frame_path)
        if frame is None:
            continue
        frame = draw_frame(frame, box, centers_so_far)
        cv2.imwrite(tmp_dir + '/frame_' + str(i).zfill(4) + '.jpg', frame)

    out_path = PROJECT_DIR + '/output_videos/' + video_name + '_tracked.mp4'
    os.system('ffmpeg -y -framerate ' + str(OUTPUT_FPS)
              + ' -i "' + tmp_dir + '/frame_%04d.jpg"'
              + ' -c:v libx264 -pix_fmt yuv420p "' + out_path + '"'
              + ' -hide_banner -loglevel error')
    print('Output video saved:', out_path)

print('Done.')


## 9. Upload to Hugging Face


In [ ]:
from huggingface_hub import login
from datasets import Dataset
import pandas as pd
from google.colab import userdata

login(token=userdata.get('HF_Token'))
HF_REPO = 'rolandchi/introai-drone-detections-assignment3'

records = []
for video_name, frame_box_map in detection_log.items():
    for frame_path, boxes in frame_box_map.items():
        for box in boxes:
            records.append({
                'video': video_name,
                'frame': os.path.basename(frame_path),
                'class': box['class'],
                'conf': box['conf'],
                'x1': box['xyxy'][0], 'y1': box['xyxy'][1],
                'x2': box['xyxy'][2], 'y2': box['xyxy'][3],
            })

ds = Dataset.from_pandas(pd.DataFrame(records))
ds.push_to_hub(HF_REPO, private=False)
print('Dataset pushed to: https://huggingface.co/datasets/' + HF_REPO)